In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import os

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src import setup_workspace, query_to_df, ds3on1

# 1. Inicializar workspace y rutas
paths = setup_workspace()

# 2. Consultar directamente a PostgreSQL (vía DuckDB ATTACH)
#df = query_to_df("SELECT * FROM db.EuInfo;")

# 3. Análisis univariable 3-en-1
#ds3on1(df, 'happy', Desc=True, Stat=True, Graph=True)

✅ Workspace anclado en: /home/josu/Documentos/DataScienceCourse
📂 Ruta de tablas verificada: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


In [2]:
#Lets prepare the data of SurveyCsv's only for the countries that got samples on every year csv
#Also we will make a economic-geographic table with usefull data to group and be the background of a timeseries.
import duckdb as db
import os
from dotenv import load_dotenv


load_dotenv()

pg_db = os.getenv("PGDATABASE")
pg_user = os.getenv("PGUSER")
pg_pass = os.getenv("PGPASSWORD")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")


conn = db.connect()
    

conn.execute("INSTALL postgres; LOAD postgres;")
    

attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={pg_db} user={pg_user} password={pg_pass} host={pg_host} port={pg_port}' 
AS db (TYPE postgres);
"""
    

conn.execute(attach_query)


listcountries = conn.execute("""
    WITH 
c2004 AS (
    SELECT DISTINCT cntry 
    FROM read_csv_auto('data/raw/Tables//Surveys04.csv')
    WHERE cntry IS NOT NULL
),
c2014 AS (
    SELECT DISTINCT cntry 
    FROM read_csv_auto('data/raw/Tables//Surveys14.csv')
    WHERE cntry IS NOT NULL
),
c2024 AS (
    SELECT DISTINCT cntry 
    FROM read_csv_auto('data/raw/Tables//Surveys24.csv')
    WHERE cntry IS NOT NULL
)

SELECT 
    COALESCE(c1.cntry, c2.cntry, c3.cntry) AS cntry,
    (c1.cntry IS NOT NULL) AS in_2004,
    (c2.cntry IS NOT NULL) AS in_2014,
    (c3.cntry IS NOT NULL) AS in_2024
FROM c2004 c1
FULL OUTER JOIN c2014 c2 ON c1.cntry = c2.cntry
FULL OUTER JOIN c2024 c3 ON COALESCE(c1.cntry, c2.cntry) = c3.cntry
ORDER BY cntry;
""").df()
print(listcountries)

   cntry  in_2004  in_2014  in_2024
0     AT     True     True     True
1     BE     True     True     True
2     BG    False    False     True
3     CH     True    False     True
4     CZ     True     True    False
5     DE     True     True     True
6     EE     True     True     True
7     ES     True     True     True
8     FI     True     True     True
9     FR     True     True     True
10    GB     True     True     True
11    GR     True    False     True
12    HR    False    False     True
13    HU    False    False     True
14    IE     True     True     True
15    IL    False    False     True
16    IT    False    False     True
17    NL     True     True     True
18    NO     True     True     True
19    PL     True     True     True
20    PT     True     True     True
21    RS    False    False     True
22    SE     True     True     True
23    SI    False    False     True
24    SK     True    False     True


In [3]:
# Muestra de países para el estudio (incluyendo Hungría y Polonia)
paises_estudio = ['AT', 'BE', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'NL', 'NO', 'PL', 'PT', 'SE']

archivos = {
    '2004': 'data/raw/Tables/Surveys04.csv',
    '2014': 'data/raw/Tables/Surveys14.csv',
    '2024': 'data/raw/Tables/Surveys24.csv'
}

columnas_datasets = {}
paises_por_dataset = {}

for ano, ruta in archivos.items():
    # 1. Obtener los nombres de las columnas de cada CSV
    cols = conn.execute(f"SELECT * FROM read_csv_auto('{ruta}') LIMIT 0").df().columns.tolist()
    columnas_datasets[f'surveys_{ano}'] = cols
    
    # 2. Extraer los países únicos efectivamente presentes en el CSV
    paises = conn.execute(f"""
        SELECT DISTINCT cntry 
        FROM read_csv_auto('{ruta}') 
        WHERE cntry IS NOT NULL 
        ORDER BY cntry
    """).df()['cntry'].tolist()
    paises_por_dataset[ano] = paises

# Diccionario consolidado
metadatos_surveys = {
    'columnas': columnas_datasets,
    'paises_existentes': paises_por_dataset,
    'muestra_final_estudio': paises_estudio
}

# Verificación
print("--- COLUMNAS DETECTADAS ---")
for k, v in metadatos_surveys['columnas'].items():
    print(f"{k}: {len(v)} columnas encontradas -> {v[:5]}...")

print("\n--- COBERTURA DE HUNGRÍA (HU) Y POLONIA (PL) ---")
for ano in ['2004', '2014', '2024']:
    hu_ok = 'HU' in metadatos_surveys['paises_existentes'][ano]
    pl_ok = 'PL' in metadatos_surveys['paises_existentes'][ano]
    print(f"Año {ano} -> HU: {hu_ok} | PL: {pl_ok}")

--- COLUMNAS DETECTADAS ---
surveys_2004: 35 columnas encontradas -> ['name', 'essround', 'edition', 'proddate', 'idno']...
surveys_2014: 34 columnas encontradas -> ['name', 'essround', 'edition', 'proddate', 'idno']...
surveys_2024: 44 columnas encontradas -> ['name', 'essround', 'edition', 'proddate', 'idno']...

--- COBERTURA DE HUNGRÍA (HU) Y POLONIA (PL) ---
Año 2004 -> HU: False | PL: True
Año 2014 -> HU: False | PL: True
Año 2024 -> HU: True | PL: True


In [ ]:
# Mapeo de rutas para los archivos base y los datos de Hungría
datasets = {
    '2004': {
        'base': 'data/raw/Tables/Surveys04.csv',
        'hu': 'data/raw/Tables/Surveys04_HU.csv'
    },
    '2014': {
        'base': 'data/raw/Tables/Surveys14.csv',
        'hu': 'data/raw/Tables/Surveys14_HU.csv'
    },
    '2024': {
        'base': 'data/raw/Tables/Surveys24.csv',
        'hu': 'data/raw/Tables/Surveys24_HU.csv'
    }
}

paises_estudio = ['AT', 'BE', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'NL', 'NO', 'PL', 'PT', 'SE']

for ano, rutas in datasets.items():
    query_union = f"""
    CREATE OR REPLACE TABLE db.public.ess_surveys_{ano} AS
    WITH dataset_unificado AS (
        SELECT 
            cntry, 
            happy, 
            lrscale, 
            emplno, 
            agea, 
            -- Usa pspwght directamente
            pspwght AS pswght
        FROM read_csv_auto('{rutas['base']}')
        
        UNION ALL
        
        SELECT 
            cntry, 
            happy, 
            lrscale, 
            emplno, 
            agea, 
            pspwght AS pswght
        FROM read_csv_auto('{rutas['hu']}')
    )
    SELECT * 
    FROM dataset_unificado
    WHERE cntry IN ({str(paises_estudio)[1:-1]});
    """
    
    conn.execute(query_union)
    print(f"Tabla 'ess_surveys_{ano}' creada en PostgreSQL (con HU integrada).")

In [4]:
# Script para verificar el recuento de respuestas (value_counts) por país y año
import pandas as pd

paises_estudio = ['AT', 'BE', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'NL', 'NO', 'PL', 'PT', 'SE']

# Consulta consolidada que une los recuentos de las 3 tablas en formato comparativo
query_counts = """
SELECT 
    COALESCE(t1.cntry, t2.cntry, t3.cntry) AS cntry,
    COALESCE(t1.total_2004, 0) AS n_2004,
    COALESCE(t2.total_2014, 0) AS n_2014,
    COALESCE(t3.total_2024, 0) AS n_2024
FROM (
    SELECT cntry, COUNT(*) AS total_2004 
    FROM db.public.ess_surveys_2004 
    GROUP BY cntry
) t1
FULL OUTER JOIN (
    SELECT cntry, COUNT(*) AS total_2014 
    FROM db.public.ess_surveys_2014 
    GROUP BY cntry
) t2 ON t1.cntry = t2.cntry
FULL OUTER JOIN (
    SELECT cntry, COUNT(*) AS total_2024 
    FROM db.public.ess_surveys_2024 
    GROUP BY cntry
) t3 ON COALESCE(t1.cntry, t2.cntry) = t3.cntry
ORDER BY cntry;
"""

# Ejecutar y mostrar como DataFrame de Pandas
df_value_counts = conn.execute(query_counts).df()

print("=== TAMAÑO MUESTRAL POR PAÍS Y AÑO (VALUE COUNTS) ===")
print(df_value_counts.to_string(index=False))

# Verificación rápida del total de registros por año
print("\n=== TOTAL DE REGISTROS CONSOLIDADOS ===")
print(f"2004: {df_value_counts['n_2004'].sum():,} respuestas")
print(f"2014: {df_value_counts['n_2014'].sum():,} respuestas")
print(f"2024: {df_value_counts['n_2024'].sum():,} respuestas")

=== TAMAÑO MUESTRAL POR PAÍS Y AÑO (VALUE COUNTS) ===
cntry  n_2004  n_2014  n_2024
   AT    2256    1795    2354
   BE    1778    1769    1594
   DE    2870    3045    2420
   EE    1989    2051    1293
   ES    1663    1925    1844
   FI    2022    2087    1563
   FR    1806    1917    1771
   GB    1897    2264    1684
   HU    1498    1698    4236
   IE    2286    2390    2017
   NL    1881    1919    1695
   NO    1760    1436    1337
   PL    1716    1615    1442
   PT    2052    1265    1373
   SE    1948    1791    1230

=== TOTAL DE REGISTROS CONSOLIDADOS ===
2004: 29,422 respuestas
2014: 28,967 respuestas
2024: 27,853 respuestas


In [5]:
#Now lets extract economic and regional data from our country study list:
#Our economic data will be the absolute register for the years 2004,2014,2024
import pandas as pd
import duckdb as db
import os
from dotenv import load_dotenv

import pandas as pd
import numpy as np

# 1. Definición de la muestra y macroregiones
paises_estudio = ['AT', 'BE', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'NL', 'NO', 'PL', 'PT', 'SE']
anios = [2004, 2014, 2024]

mapa_regiones = {
    'AT': 'western', 'BE': 'western', 'DE': 'western', 'FR': 'western', 
    'GB': 'western', 'IE': 'western', 'NL': 'western',
    'EE': 'northern', 'FI': 'northern', 'NO': 'northern', 'SE': 'northern',
    'ES': 'southern', 'PT': 'southern',
    'HU': 'eastern', 'PL': 'eastern'
}

# 2. Generar el producto cartesiano (15 países x 3 años = 45 filas)
index = pd.MultiIndex.from_product([paises_estudio, anios], names=['geo', 'year'])
df_macro = pd.DataFrame(index=index).reset_index()

# 3. Asignar macroregión
df_macro['macro_region'] = df_macro['geo'].map(mapa_regiones)

# 4. Inicializar columnas con NaN para ir "cocinando" los datos
columnas_macro = [
    'reglangdemographypct', # Demografía / Población
    'unemploymentpct',      # Tasa de desempleo (%)
    'govdebtgdpabspct',     # Deuda pública (% del PIB)
    'povertypct',           # Riesgo de pobreza (%)
    'ginicoefpct',          # Índice de Gini
    'gdpcapitapct',         # PIB per cápita (USD / EUR constantes)
    'disposableincomepct',  # Renta disponible per cápita
    'inflationpct'          # Inflación (%)
]

for col in columnas_macro:
    df_macro[col] = np.nan

print(f"Esqueleto creado: {df_macro.shape}") # (45, 11)

Esqueleto creado: (45, 11)


In [7]:
import os
import time
import requests
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET

# 1. Definición de la muestra y macroregiones
paises_estudio = ['AT', 'BE', 'DE', 'EE', 'ES', 'FI', 'FR', 'GB', 'HU', 'IE', 'NL', 'NO', 'PL', 'PT', 'SE']
anios = [2004, 2014, 2024]

mapa_regiones = {
    'AT': 'western', 'BE': 'western', 'DE': 'western', 'FR': 'western', 
    'GB': 'western', 'IE': 'western', 'NL': 'western',
    'EE': 'northern', 'FI': 'northern', 'NO': 'northern', 'SE': 'northern',
    'ES': 'southern', 'PT': 'southern',
    'HU': 'eastern', 'PL': 'eastern'
}

wb_indicators = {
    'NY.GDP.PCAP.CD': 'gdpcapitapct',      # PIB per cápita (USD)
    'SL.UEM.TOTL.ZS': 'unemploymentpct',   # Desempleo (%)
    'FP.CPI.TOTL.ZG': 'inflationpct',      # Inflación (%)
    'SP.POP.TOTL': 'reglangdemographypct', # Población
    'SI.POV.GINI': 'ginicoefpct'           # Gini
}

# Crear directorio local si no existe
dir_xml = 'data/raw_xml'
os.makedirs(dir_xml, exist_ok=True)

# ---------------------------------------------------------
# FASE 1: Descarga con Delay de 5 Segundos
# ---------------------------------------------------------
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Connection': 'close' # Forzar el cierre explícito para liberar el socket
})

print("=== FASE 1: DESCARGA PAUSADA (1 PETICIÓN CADA 5 SEGUNDOS) ===")

total_requests = len(paises_estudio) * len(wb_indicators)
counter = 0

for country in paises_estudio:
    for code, col_name in wb_indicators.items():
        counter += 1
        file_path = f"{dir_xml}/{country}_{col_name}.xml"
        
        # Solo descargar si el archivo aún no se ha guardado en disco
        if not os.path.exists(file_path):
            url = f"https://api.worldbank.org/v2/country/{country}/indicator/{code}?date=2004:2024&per_page=1000"
            
            try:
                res = session.get(url, timeout=20)
                if res.status_code == 200:
                    with open(file_path, 'wb') as f:
                        f.write(res.content)
                        f.flush()
                    print(f"[{counter}/{total_requests}] Guardado OK: {country}_{col_name}.xml")
                else:
                    print(f"[{counter}/{total_requests}] HTTP {res.status_code} en {country} - {col_name}")
            except Exception as e:
                print(f"[{counter}/{total_requests}] Error en {country} - {col_name}: {e}")
            
            # PAUSA CRUCIAL: 5 SEGUNDOS ENTRE REQUESTS
            time.sleep(5.0)
        else:
            print(f"[{counter}/{total_requests}] Ya existe en disco: {country}_{col_name}.xml")

# ---------------------------------------------------------
# FASE 2: Parseo Local y Carga en Pandas
# ---------------------------------------------------------
print("\n=== FASE 2: PARSEO LOCAL DE ARCHIVOS XML ===")
data_list = []

for country in paises_estudio:
    for code, col_name in wb_indicators.items():
        file_path = f"{dir_xml}/{country}_{col_name}.xml"
        
        if os.path.exists(file_path):
            try:
                tree = ET.parse(file_path)
                root = tree.getroot()
                
                for elem in root.iter():
                    if elem.tag.endswith('data'):
                        date_val = None
                        num_val = None
                        
                        for child in elem:
                            if child.tag.endswith('date'):
                                date_val = child.text
                            elif child.tag.endswith('value'):
                                num_val = child.text
                        
                        if date_val and num_val:
                            yr = int(date_val)
                            if yr in anios:
                                data_list.append({
                                    'geo': country,
                                    'year': yr,
                                    'variable': col_name,
                                    'val': float(num_val)
                                })
            except Exception as e:
                print(f"Error parseando {file_path}: {e}")

# ---------------------------------------------------------
# FASE 3: Inyección al DataFrame en Memoria
# ---------------------------------------------------------
index = pd.MultiIndex.from_product([paises_estudio, anios], names=['geo', 'year'])
df_macro = pd.DataFrame(index=index).reset_index()
df_macro['macro_region'] = df_macro['geo'].map(mapa_regiones)

df_parsed = pd.DataFrame(data_list)
if not df_parsed.empty:
    df_pivoted = df_parsed.pivot(index=['geo', 'year'], columns='variable', values='val').reset_index()
    
    df_macro = df_macro.set_index(['geo', 'year'])
    df_pivoted = df_pivoted.set_index(['geo', 'year'])
    df_macro.update(df_pivoted)
    df_macro = df_macro.reset_index()

print("\n=== DATAFRAME COMPLETO EN MEMORIA ===")
print(df_macro[['geo', 'year', 'macro_region', 'gdpcapitapct', 'unemploymentpct', 'inflationpct']].head(12))

=== FASE 1: DESCARGA PAUSADA (1 PETICIÓN CADA 5 SEGUNDOS) ===
[1/75] Ya existe en disco: AT_gdpcapitapct.xml
[2/75] Ya existe en disco: AT_unemploymentpct.xml
[3/75] Ya existe en disco: AT_inflationpct.xml
[4/75] Ya existe en disco: AT_reglangdemographypct.xml
[5/75] Ya existe en disco: AT_ginicoefpct.xml
[6/75] Ya existe en disco: BE_gdpcapitapct.xml
[7/75] Ya existe en disco: BE_unemploymentpct.xml
[8/75] Ya existe en disco: BE_inflationpct.xml
[9/75] Ya existe en disco: BE_reglangdemographypct.xml
[10/75] Ya existe en disco: BE_ginicoefpct.xml
[11/75] Ya existe en disco: DE_gdpcapitapct.xml
[12/75] Ya existe en disco: DE_unemploymentpct.xml
[13/75] Ya existe en disco: DE_inflationpct.xml
[14/75] Ya existe en disco: DE_reglangdemographypct.xml
[15/75] Guardado OK: DE_ginicoefpct.xml
[16/75] Ya existe en disco: EE_gdpcapitapct.xml
[17/75] Ya existe en disco: EE_unemploymentpct.xml
[18/75] Ya existe en disco: EE_inflationpct.xml
[19/75] Guardado OK: EE_reglangdemographypct.xml
[20/75] 

KeyError: "['gdpcapitapct', 'unemploymentpct', 'inflationpct'] not in index"

In [11]:
# We have all the data except GiniCoef of 2024. We imput it manually on df_pivoted:
# Diccionario de respaldo exclusivo para el Coeficiente de Gini (%) en 2024 (Eurostat / OCDE)
gini_2024_map = {
    'AT': 30.2, 'BE': 27.5, 'DE': 29.4, 'EE': 31.8, 'ES': 33.0,
    'FI': 26.8, 'FR': 31.5, 'GB': 35.1, 'HU': 29.0, 'IE': 29.2,
    'NL': 28.1, 'NO': 27.6, 'PL': 28.5, 'PT': 32.0, 'SE': 27.6
}

# 1. Crear la máscara sobre el nivel 'year' del MultiIndex
mask_2024 = (df_pivoted.index.get_level_values('year') == 2024)

# 2. Extraer los códigos 'geo' del MultiIndex para las filas de 2024
geos_2024 = df_pivoted.index.get_level_values('geo')[mask_2024]

# 3. Asignar los valores mapeados directamente sobre ginicoefpct
df_pivoted.loc[mask_2024, 'ginicoefpct'] = geos_2024.map(gini_2024_map).values

print("=== NULOS EN GINICOEFPCT TRAS EL PARCHE EN DF_PIVOTED ===")
print(df_pivoted['ginicoefpct'].isna().sum())

=== NULOS EN GINICOEFPCT TRAS EL PARCHE EN DF_PIVOTED ===
0


In [12]:
df_pivoted.head(12)

variable  gdpcapitapct  ginicoefpct  inflationpct  reglangdemographypct  \
geo year                                                                  
AT  2004  36614.250653         29.8      2.061206             8171966.0   
    2014  51314.972262         30.5      1.605812             8546356.0   
    2024  58268.878765         30.2      2.937916             9177982.0   
BE  2004  35429.407793         30.5      2.097285            10421137.0   
    2014  47995.778696         28.1      0.349556            11209057.0   
    2024  56581.937564         27.5      3.144266            11858610.0   
DE  2004  34566.735914         30.1      1.665737            82516260.0   
    2014  48959.599120         30.8      0.906794            80982500.0   
    2024  56103.732318         29.4      2.256498            83516593.0   
EE  2004   8916.072827         33.6      3.048029             1362550.0   
    2014  20581.789899         34.6     -0.110672             1314545.0   
    2024  31428.354782         31.8      3.522465             1372341.0   

variable  unemploymentpct  
geo year                   
AT  2004            5.969  
    2014            5.674  
    2024            5.200  
BE  2004            7.363  
    2014            8.523  
    2024            5.700  
DE  2004           10.727  
    2014            4.979  
    2024            3.400  
EE  2004           10.248  
    2014            7.352  
    2024            7.600

In [22]:
import os
import duckdb as db
from dotenv import load_dotenv

# 1. Cargar variables de entorno
load_dotenv()

pg_db = os.getenv("PGDATABASE")
pg_user = os.getenv("PGUSER")
pg_pass = os.getenv("PGPASSWORD")
pg_host = os.getenv("PGHOST")
pg_port = os.getenv("PGPORT")

# 2. Inicializar conexión única en DuckDB
conn = db.connect()

# Instalar y cargar extensión de Postgres
conn.execute("INSTALL postgres; LOAD postgres;")

# Conectar/Adjuntar base de datos externa de Postgres
attach_query = f"""
ATTACH IF NOT EXISTS 'dbname={pg_db} user={pg_user} password={pg_pass} host={pg_host} port={pg_port}' 
AS db (TYPE postgres);
"""
conn.execute(attach_query)

# 3. Mapeo de regiones y reestructuración del DataFrame
mapa_regiones = {
    'AT': 'western', 'BE': 'western', 'DE': 'western', 'FR': 'western', 
    'GB': 'western', 'IE': 'western', 'NL': 'western',
    'EE': 'northern', 'FI': 'northern', 'NO': 'northern', 'SE': 'northern',
    'ES': 'southern', 'PT': 'southern',
    'HU': 'eastern', 'PL': 'eastern'
}

if 'geo' not in df_pivoted.columns:
    df_pivoted = df_pivoted.reset_index()

df_pivoted.columns.name = None
df_pivoted['macro_region'] = df_pivoted['geo'].map(mapa_regiones)

columnas_ordenadas = [
    'geo', 
    'year', 
    'macro_region', 
    'gdpcapitapct', 
    'ginicoefpct', 
    'inflationpct', 
    'reglangdemographypct', 
    'unemploymentpct'
]

df_final = df_pivoted[columnas_ordenadas].copy()

# 4. Crear o reemplazar la tabla usando 'conn'
# Usa 'CREATE OR REPLACE TABLE db.macro_panel' si quieres escribirla directamente en Postgres
conn.execute("CREATE OR REPLACE TABLE db.macro_panel AS SELECT * FROM df_final")



In [23]:
# 5. Verificación
print("=== ESQUEMA EN DUCKDB ===")
print(conn.execute("DESCRIBE db.macro_panel").df())

print("\n=== VISTA PREVIA DUCKDB (PRIMERAS FILAS) ===")
print(conn.execute("SELECT * FROM db.macro_panel LIMIT 6").df())

=== ESQUEMA EN DUCKDB ===
            column_name column_type null   key default extra
0                   geo     VARCHAR  YES  None    None  None
1                  year      BIGINT  YES  None    None  None
2          macro_region     VARCHAR  YES  None    None  None
3          gdpcapitapct      DOUBLE  YES  None    None  None
4           ginicoefpct      DOUBLE  YES  None    None  None
5          inflationpct      DOUBLE  YES  None    None  None
6  reglangdemographypct      DOUBLE  YES  None    None  None
7       unemploymentpct      DOUBLE  YES  None    None  None

=== VISTA PREVIA DUCKDB (PRIMERAS FILAS) ===
  geo  year macro_region  gdpcapitapct  ginicoefpct  inflationpct  \
0  AT  2004      western  36614.250653         29.8      2.061206   
1  AT  2014      western  51314.972262         30.5      1.605812   
2  AT  2024      western  58268.878765         30.2      2.937916   
3  BE  2004      western  35429.407793         30.5      2.097285   
4  BE  2014      western  47995.77

In [ ]:
#Apis are usefull but if you dont have special rights on those urls u got timmed out so early.
#Some times is better make the dictionary and fulfill the data manually or datascrapping from internet tables.